# v4r5 QLoRA training (Colab T4)

This notebook trains Qwen3-4B on the clean, broad v4r5 dataset. It intentionally stops after decode calibration. Do not run litmus or `blind_v4r5` until the decoding setting is locked.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select the Colab T4 GPU kernel first.'
print(torch.cuda.get_device_name(0))
!nvidia-smi

Tesla T4
Sat Jul 11 21:41:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------------------------

Tesla T4
Sun Jul 12 19:27:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P0             26W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------------------------

In [ ]:
!pip install -q unsloth trl datasets textstat

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 12.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 116.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.3 MB/s eta 0:00:00:00:010:02m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

## Train v4r5

Commit and push the finished dataset and code before running this cell. Colab clones GitHub, so uncommitted local files are invisible to the remote kernel.

In [ ]:
%cd /content
![ -d SmallLearningModel ] || git clone https://github.com/Elitelord/SmallLearningModel.git
%cd SmallLearningModel
!git pull --ff-only
!python -m data.audit_v4r3 data/v4/gold_v4_r5.jsonl --accuracy-gate clean-v2 --forbid-targeted-v4r3
!python -m train.qlora_train \
    --data data/v4/gold_v4_r5.jsonl \
    --adapter-name v4r5 \
    --epochs 2 \
    --lora-r 16 --lora-alpha 32 \
    --batch-size 8 --grad-accum 2 --lr 1e-4

/content
Cloning into 'SmallLearningModel'...
remote: Enumerating objects: 319, done.
remote: Counting objects: 100% (319/319), done.
remote: Compressing objects: 100% (216/216), done.
remote: Total 319 (delta 162), reused 241 (delta 91), pack-reused 0 (from 0)
Receiving objects: 100% (319/319), 1.80 MiB | 16.57 MiB/s, done.
Resolving deltas: 100% (162/162), done.
/content/SmallLearningModel
Already up to date.
{
  "path": "data/v4/gold_v4_r5.jsonl",
  "records": 402,
  "unique_prompts": 402,
  "target": {
    "name": "v4r3",
    "fk_band": [
      3.3,
      5.0
    ],
    "ari_band": [
      3.8,
      6.2
    ],
    "disp_max": 1.1,
    "max_sentence_fk": 7.0
  },
  "min_sentences": 4,
  "max_sentences": 6,
  "accuracy_gate": "clean-v2",
  "forbid_targeted_v4r3": true,
  "passed": true,
  "failures": [],
  "failure_count": 0
}
Loaded 402 training records from data/v4/gold_v4_r5.jsonl
CUDA detected -> Unsloth 4-bit QLoRA path
🦥 Unsloth: Will patch your computer to enable 2x faster fr

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
backup_dir = Path('/content/drive/MyDrive/SmallLearningModel/v4r5')
backup_dir.mkdir(parents=True, exist_ok=True)
shutil.copytree(Path('train/adapters/v4r5'), backup_dir / 'adapter', dirs_exist_ok=True)
print(f'Adapter backed up to {backup_dir / "adapter"}')

Mounted at /content/drive
Adapter backed up to /content/drive/MyDrive/SmallLearningModel/v4r5/adapter


## Decode calibration

This uses only `calibration_v4r5`, never the litmus or blind holdout. Choose temperature by average readability across every seed, not by the best individual seed.

In [ ]:
!python -m eval.tuned_sweep \
    --adapter train/adapters/v4r5 \
    --eval-key calibration_v4r5 \
    --temperatures 0,0.3,0.7 --seeds 0,1,2 \
    --top-settings 7 \
    --out eval/v4r5_decode_calibration.json

config.json: 100% 726/726 [00:00<00:00, 3.84MB/s]
tokenizer_config.json: 100% 9.73k/9.73k [00:00<00:00, 5.47MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 84.5MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 123MB/s]
tokenizer.json: 100% 11.4M/11.4M [00:01<00:00, 7.98MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
model.safetensors.index.json: 100% 32.8k/32.8k [00:00<00:00, 83.0MB/s]
Fetching 3 files: 100% 3/3 [03:58<00:00, 79.48s/it] 
Download complete: 100% 8.04G/8.04G [03:58<00:00, 33.7MB/s]                
Loading weights: 100% 398/398 [00:31<00:00, 12.59it/s]
generation_config.json: 100% 239/239 [00:00<00:00, 1.04MB/s]

=== t0p0_s0 ===
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[1/24] read=False penalty=0.14 What makes a rubber ball bounce?
[2/24] read=False penalt

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
backup_dir = Path('/content/drive/MyDrive/SmallLearningModel/v4r5')
backup_dir.mkdir(parents=True, exist_ok=True)
for result_path in Path('eval').glob('v4r5_decode_calibration*.json'):
    shutil.copy2(result_path, backup_dir / result_path.name)
print(f'Calibration files backed up to {backup_dir}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Calibration files backed up to /content/drive/MyDrive/SmallLearningModel/v4r5


## Calibration reviewed

Temperature `0.7` won on average readability across all seeds (10.33/24 versus 9/24 at 0.3 and 8/24 at 0). The setting is now locked. Run the one-shot litmus regression below, but do not run `blind_v4r5` yet.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')
repo_dir = Path('/content/SmallLearningModel')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Elitelord/SmallLearningModel.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
backup_dir = Path('/content/drive/MyDrive/SmallLearningModel/v4r5')
adapter_dir = Path('train/adapters/v4r5')
if not adapter_dir.exists():
    shutil.copytree(backup_dir / 'adapter', adapter_dir, dirs_exist_ok=True)

subprocess.run([
    sys.executable, '-m', 'eval.tuned_sweep',
    '--adapter', str(adapter_dir),
    '--eval-key', 'eval_litmus', '--final-eval',
    '--temperatures', '0.7', '--seeds', '0',
    '--top-settings', '1',
    '--out', 'eval/v4r5_decode_litmus.json',
], check=True)

for result_path in Path('eval').glob('v4r5_decode_litmus*.json'):
    shutil.copy2(result_path, backup_dir / result_path.name)
print(f'Litmus files backed up to {backup_dir}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Litmus files backed up to /content/drive/MyDrive/SmallLearningModel/v4r5


## Stop before blind evaluation

Copy the two litmus JSON files back into local `eval/` and run accuracy-v2 judging. Only run `blind_v4r5` after the litmus regression is recorded and no further model changes are planned.

## Controlled r32 capacity ablation

v4r5 missed the litmus target, so the blind holdout remains sealed. This run changes only LoRA capacity from r16/a32 to r32/a64 while keeping the same clean data, two epochs, and learning rate `1e-4`.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')
repo_dir = Path('/content/SmallLearningModel')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Elitelord/SmallLearningModel.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
subprocess.run(['git', 'pull', '--ff-only'], check=True)
subprocess.run([
    sys.executable, '-m', 'data.audit_v4r3', 'data/v4/gold_v4_r5.jsonl',
    '--accuracy-gate', 'clean-v2', '--forbid-targeted-v4r3',
], check=True)

adapter_dir = Path('train/adapters/v4r5_r32_e2')
if adapter_dir.exists():
    raise FileExistsError(f'{adapter_dir} already exists; refusing to overwrite it')
subprocess.run([
    sys.executable, '-m', 'train.qlora_train',
    '--data', 'data/v4/gold_v4_r5.jsonl',
    '--adapter-name', 'v4r5_r32_e2',
    '--epochs', '2', '--lora-r', '32', '--lora-alpha', '64',
    '--batch-size', '8', '--grad-accum', '2', '--lr', '1e-4',
], check=True)

backup_dir = Path('/content/drive/MyDrive/SmallLearningModel/v4r5_r32_e2')
backup_dir.mkdir(parents=True, exist_ok=True)
shutil.copytree(adapter_dir, backup_dir / 'adapter', dirs_exist_ok=True)
print(f'r32 adapter backed up to {backup_dir / "adapter"}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
r32 adapter backed up to /content/drive/MyDrive/SmallLearningModel/v4r5_r32_e2/adapter


In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')
repo_dir = Path('/content/SmallLearningModel')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Elitelord/SmallLearningModel.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
backup_dir = Path('/content/drive/MyDrive/SmallLearningModel/v4r5_r32_e2')
adapter_dir = Path('train/adapters/v4r5_r32_e2')
if not adapter_dir.exists():
    shutil.copytree(backup_dir / 'adapter', adapter_dir, dirs_exist_ok=True)

output_path = Path('eval/v4r5_r32_e2_decode_calibration.json')
top_path = Path('eval/v4r5_r32_e2_decode_calibration.top.json')
for local_path in (output_path, top_path):
    drive_path = backup_dir / local_path.name
    if drive_path.exists() and not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_path, local_path)
if output_path.exists() and top_path.exists():
    print('Calibration files already exist; skipping duplicate generation.')
else:
    command = [
        sys.executable, '-u', '-m', 'eval.tuned_sweep',
        '--adapter', str(adapter_dir),
        '--eval-key', 'calibration_v4r5',
        '--temperatures', '0,0.3,0.7', '--seeds', '0,1,2',
        '--top-settings', '7',
        '--out', str(output_path),
    ]
    print('Starting r32 calibration; model loading may be silent for several minutes.', flush=True)
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
for result_path in Path('eval').glob('v4r5_r32_e2_decode_calibration*.json'):
    shutil.copy2(result_path, backup_dir / result_path.name)
print(f'r32 calibration files backed up to {backup_dir}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Calibration files already exist; skipping duplicate generation.
r32 calibration files backed up to /content/drive/MyDrive/SmallLearningModel/v4r5_r32_e2


In [ ]:
!pkill -f "eval.tuned_sweep" || true
!nvidia-smi

^C
Sun Jul 12 01:03:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   65C    P0             31W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+--

## Stop after r32 calibration

r32 passed the predeclared gate: temperature `0.3` averaged 13/24 readability versus r16's 10.33/24. The fixed one-shot litmus run below uses temperature `0.3` and seed `0`.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')
repo_dir = Path('/content/SmallLearningModel')
os.chdir(repo_dir)
backup_dir = Path('/content/drive/MyDrive/SmallLearningModel/v4r5_r32_e2')
adapter_dir = Path('train/adapters/v4r5_r32_e2')
if not adapter_dir.exists():
    shutil.copytree(backup_dir / 'adapter', adapter_dir, dirs_exist_ok=True)
output_path = Path('eval/v4r5_r32_e2_decode_litmus.json')
top_path = Path('eval/v4r5_r32_e2_decode_litmus.top.json')

command = [
    sys.executable, '-u', '-m', 'eval.tuned_sweep',
    '--adapter', str(adapter_dir),
    '--eval-key', 'eval_litmus', '--final-eval',
    '--temperatures', '0.3', '--seeds', '0',
    '--top-settings', '1', '--out', str(output_path),
]
process = subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, command)
for result_path in (output_path, top_path):
    shutil.copy2(result_path, backup_dir / result_path.name)
print(f'r32 litmus files backed up to {backup_dir}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
`torch_dtype` is deprecated! Use `dtype` instead!
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).

Loading weights: 100%|██████████| 398/398 [00:32<00:00, 12.31it/s]

=== t0p3_s0 ===
[1/12] read=True penalty=0.0 Why is the sky blue?
[2/12] read=False penalty=0.0 How do plants make their own food?
[3/12] read=True penalty=0.0 Why do we have day and night?
[4/12] read=False penalty=0.03 What makes ice melt?
[5/12] read=False penalty=1.31 How do magnets work?
[6/12] read=False penalty=0.01 Why do things fall to the ground?
[7/12] read=True penalty=0.0 Where does a puddle go when it dries up?
[8/12] read=False penalty=0.9 Why do we have seasons?
[9/12] read=True penalty=0.0 How do our lungs help us breathe?
[10/12] read=True penalty=0.0 What makes a rainbow?
[11/12] read=False penalty=0.

## Stop after r32 litmus

Copy the two r32 litmus JSON files into local `eval/` for accuracy-v2 judging. Keep `blind_v4r5` sealed.

## Controlled r16 three-epoch ablation

This changes only training duration from the original v4r5 run: r16/a32, learning rate `1e-4`, and the same 402 clean records remain fixed, while epochs increase from two to three.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')
repo_dir = Path('/content/SmallLearningModel')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Elitelord/SmallLearningModel.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
subprocess.run(['git', 'pull', '--ff-only'], check=True)
subprocess.run([
    sys.executable, '-m', 'data.audit_v4r3', 'data/v4/gold_v4_r5.jsonl',
    '--accuracy-gate', 'clean-v2', '--forbid-targeted-v4r3',
], check=True)

adapter_dir = Path('train/adapters/v4r5_r16_e3')
if adapter_dir.exists():
    raise FileExistsError(f'{adapter_dir} already exists; refusing to overwrite it')
subprocess.run([
    sys.executable, '-m', 'train.qlora_train',
    '--data', 'data/v4/gold_v4_r5.jsonl',
    '--adapter-name', 'v4r5_r16_e3',
    '--epochs', '3', '--lora-r', '16', '--lora-alpha', '32',
    '--batch-size', '8', '--grad-accum', '2', '--lr', '1e-4',
], check=True)

backup_dir = Path('/content/drive/MyDrive/SmallLearningModel/v4r5_r16_e3')
backup_dir.mkdir(parents=True, exist_ok=True)
shutil.copytree(adapter_dir, backup_dir / 'adapter', dirs_exist_ok=True)
print(f'r16/e3 adapter backed up to {backup_dir / "adapter"}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
r16/e3 adapter backed up to /content/drive/MyDrive/SmallLearningModel/v4r5_r16_e3/adapter


In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')
repo_dir = Path('/content/SmallLearningModel')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Elitelord/SmallLearningModel.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
try:
    import transformers, peft, bitsandbytes, textstat
except ImportError:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'transformers', 'peft', 'bitsandbytes', 'textstat',
    ], check=True)
backup_dir = Path('/content/drive/MyDrive/SmallLearningModel/v4r5_r16_e3')
adapter_dir = Path('train/adapters/v4r5_r16_e3')
if not adapter_dir.exists():
    shutil.copytree(backup_dir / 'adapter', adapter_dir, dirs_exist_ok=True)
output_path = Path('eval/v4r5_r16_e3_decode_calibration.json')
top_path = Path('eval/v4r5_r16_e3_decode_calibration.top.json')
for local_path in (output_path, top_path):
    drive_path = backup_dir / local_path.name
    if drive_path.exists() and not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_path, local_path)

if output_path.exists() and top_path.exists():
    print('Calibration files already exist; skipping duplicate generation.')
else:
    command = [
        sys.executable, '-u', '-m', 'eval.tuned_sweep',
        '--adapter', str(adapter_dir),
        '--eval-key', 'calibration_v4r5',
        '--temperatures', '0,0.3,0.7', '--seeds', '0,1,2',
        '--top-settings', '7', '--out', str(output_path),
    ]
    print('Starting r16/e3 calibration; model loading may be silent for several minutes.', flush=True)
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
for result_path in (output_path, top_path):
    shutil.copy2(result_path, backup_dir / result_path.name)
print(f'r16/e3 calibration files backed up to {backup_dir}')

Mounted at /content/drive
Calibration files already exist; skipping duplicate generation.
r16/e3 calibration files backed up to /content/drive/MyDrive/SmallLearningModel/v4r5_r16_e3


## r16/e3 calibration passed

Deterministic temperature `0` reached 13/24 readability, beating the original r16/e2 result of 10.33/24. The fixed one-shot litmus run below uses temperature `0` and seed `0`. Keep `blind_v4r5` sealed.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')
repo_dir = Path('/content/SmallLearningModel')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Elitelord/SmallLearningModel.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
try:
    import transformers, peft, bitsandbytes, textstat
except ImportError:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'transformers', 'peft', 'bitsandbytes', 'textstat',
    ], check=True)
backup_dir = Path('/content/drive/MyDrive/SmallLearningModel/v4r5_r16_e3')
adapter_dir = Path('train/adapters/v4r5_r16_e3')
if not adapter_dir.exists():
    shutil.copytree(backup_dir / 'adapter', adapter_dir, dirs_exist_ok=True)
output_path = Path('eval/v4r5_r16_e3_decode_litmus.json')
top_path = Path('eval/v4r5_r16_e3_decode_litmus.top.json')
for local_path in (output_path, top_path):
    drive_path = backup_dir / local_path.name
    if drive_path.exists() and not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_path, local_path)

if output_path.exists() and top_path.exists():
    print('Litmus files already exist; refusing a duplicate run.')
else:
    command = [
        sys.executable, '-u', '-m', 'eval.tuned_sweep',
        '--adapter', str(adapter_dir),
        '--eval-key', 'eval_litmus', '--final-eval',
        '--temperatures', '0', '--seeds', '0',
        '--top-settings', '1', '--out', str(output_path),
    ]
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
for result_path in (output_path, top_path):
    shutil.copy2(result_path, backup_dir / result_path.name)
print(f'r16/e3 litmus files backed up to {backup_dir}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

Fetching 3 files: 100%|██████████| 3/3 [01:43<00:00, 34.62s/it]

Loading weights: 100%|██████████| 398/398 [00:31<00:00, 12.67it/s]

=== t0p0_s0 ===
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[1/12] read=False penalty=0.08 Why is the sky blue?
[2/12] read=False penalty=0.87 How do plants make their own food?
[3/12] read=True penalty=0.0 Why do we have day and night?
[4/12] read=False penalty=0.68 What makes ice melt?
[5/12] read=True penalty=0.0 How do magnets work?
[6/12] read=False penalty=0.84 Why do things fall to the ground?
[7/12] read=False penalty=0.61 Where does a puddle go when i

## Stop after r16/e3 litmus

Copy the two r16/e3 litmus JSON files into local `eval/`. Keep `blind_v4r5` sealed.

## Controlled v4r6 mixed replay

Run this section only after the local accuracy-v2 audits and `data.build_v4r6_mix` succeed. The 400-record dataset combines all 98 clean tight v4r2 accuracy anchors, 102 clean tight v4r4 readability records, and 200 clean v4r5 targets. Training remains r16/a32 for two epochs at learning rate `1e-4`.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')
repo_dir = Path('/content/SmallLearningModel_v4r6')
if not repo_dir.exists():
    subprocess.run([
        'git', 'clone', '--branch', 'main', '--single-branch',
        'https://github.com/Elitelord/SmallLearningModel.git', str(repo_dir),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo_dir), 'reset', '--hard', 'origin/main'], check=True)
os.chdir(repo_dir)
dataset_path = Path('data/v4/gold_v4_r6.jsonl')
stats_path = Path('data/v4/gold_v4_r6.stats.json')
if not dataset_path.exists() or not stats_path.exists():
    raise FileNotFoundError('Push the validated v4r6 dataset and stats before training')
stats = json.loads(stats_path.read_text(encoding='utf-8'))
dataset_bytes = dataset_path.read_bytes().replace(b'\r\n', b'\n').replace(b'\r', b'\n')
dataset_hash = hashlib.sha256(dataset_bytes).hexdigest()
if dataset_hash != stats.get('dataset_sha256'):
    raise ValueError(f'v4r6 dataset hash mismatch: {dataset_hash}')
try:
    import unsloth, trl, datasets, textstat
except ImportError:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'unsloth', 'trl', 'datasets', 'textstat',
    ], check=True)
except ValueError as error:
    if 'pyarrow' not in str(error).lower() and 'IpcReadOptions' not in str(error):
        raise
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
        '--no-cache-dir', '--no-deps', 'pyarrow==18.1.0',
    ], check=True)
    raise RuntimeError('PyArrow repaired. Restart the kernel, then rerun this cell.') from error
subprocess.run([
    sys.executable, '-m', 'data.audit_v4r3', str(dataset_path),
    '--accuracy-gate', 'clean-v2', '--forbid-targeted-v4r3',
], check=True)

adapter_dir = Path('train/adapters/v4r6')
if adapter_dir.exists():
    raise FileExistsError(f'{adapter_dir} already exists; refusing to overwrite it')
command = [
    sys.executable, '-m', 'train.qlora_train',
    '--data', str(dataset_path), '--adapter-name', 'v4r6',
    '--epochs', '2', '--lora-r', '16', '--lora-alpha', '32',
    '--batch-size', '8', '--grad-accum', '2', '--lr', '1e-4',
]
print('Starting v4r6 training.', flush=True)
process = subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, command)

backup_dir = Path('/content/drive/MyDrive/SmallLearningModel/v4r6')
backup_dir.mkdir(parents=True, exist_ok=True)
shutil.copytree(adapter_dir, backup_dir / 'adapter', dirs_exist_ok=True)
shutil.copy2(dataset_path, backup_dir / dataset_path.name)
shutil.copy2(stats_path, backup_dir / stats_path.name)
print(f'v4r6 adapter and dataset backed up to {backup_dir}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Starting v4r6 training.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be diff

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')
repo_dir = Path('/content/SmallLearningModel_v4r6')
if not repo_dir.exists():
    subprocess.run([
        'git', 'clone', '--branch', 'main', '--single-branch',
        'https://github.com/Elitelord/SmallLearningModel.git', str(repo_dir),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo_dir), 'reset', '--hard', 'origin/main'], check=True)
os.chdir(repo_dir)
backup_dir = Path('/content/drive/MyDrive/SmallLearningModel/v4r6')
adapter_dir = Path('train/adapters/v4r6')
if not adapter_dir.exists():
    shutil.copytree(backup_dir / 'adapter', adapter_dir, dirs_exist_ok=True)
output_path = Path('eval/v4r6_decode_calibration.json')
top_path = Path('eval/v4r6_decode_calibration.top.json')
for local_path in (output_path, top_path):
    drive_path = backup_dir / local_path.name
    if drive_path.exists() and not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_path, local_path)
if output_path.exists() and top_path.exists():
    print('v4r6 calibration files already exist; skipping duplicate generation.')
else:
    command = [
        sys.executable, '-u', '-m', 'eval.tuned_sweep',
        '--adapter', str(adapter_dir), '--eval-key', 'calibration_v4r5',
        '--temperatures', '0,0.3,0.7', '--seeds', '0,1,2',
        '--top-settings', '7', '--out', str(output_path),
    ]
    print('Starting v4r6 calibration.', flush=True)
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
for result_path in (output_path, top_path):
    shutil.copy2(result_path, backup_dir / result_path.name)
print(f'v4r6 calibration files backed up to {backup_dir}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Starting v4r6 calibration.
`torch_dtype` is deprecated! Use `dtype` instead!
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).

Loading weights: 100%|██████████| 398/398 [00:33<00:00, 11.95it/s]

=== t0p0_s0 ===
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[1/24] read=False penalty=0.46 What makes a rubber ball bounce?
[2/24] read=False penalty=0.14 How does soap lift dirt from our hands?
[3/24] read=True penalty=0.0 Why does metal feel colder than wood?
[4/24] read=False penalty=0.35 How does a thermos keep a drink warm?
[5/24] read=False penalty=0.63 How do bicycle brakes slow the wheels?
[6/24] read=False penalty=0.79 What makes static electricity build up?
[7/24] read=True penalty=0.0 How does glue h

: 

## v4r6 calibration passed

Temperature `0` reached 12/24 readability. Temperature `0.3` averaged 10.33/24 across seeds, and `0.7` averaged 10/24. The fixed development-litmus run below therefore uses temperature `0` and seed `0`. Keep `blind_v4r5` sealed.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')
repo_dir = Path('/content/SmallLearningModel_v4r6')
os.chdir(repo_dir)
backup_dir = Path('/content/drive/MyDrive/SmallLearningModel/v4r6')
adapter_dir = Path('train/adapters/v4r6')
if not adapter_dir.exists():
    shutil.copytree(backup_dir / 'adapter', adapter_dir, dirs_exist_ok=True)
output_path = Path('eval/v4r6_decode_litmus.json')
top_path = Path('eval/v4r6_decode_litmus.top.json')
for local_path in (output_path, top_path):
    drive_path = backup_dir / local_path.name
    if drive_path.exists() and not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_path, local_path)
if output_path.exists() and top_path.exists():
    print('v4r6 litmus files already exist; refusing a duplicate run.')
else:
    command = [
        sys.executable, '-u', '-m', 'eval.tuned_sweep',
        '--adapter', str(adapter_dir),
        '--eval-key', 'eval_litmus', '--final-eval',
        '--temperatures', '0', '--seeds', '0',
        '--top-settings', '1', '--out', str(output_path),
    ]
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
for result_path in (output_path, top_path):
    shutil.copy2(result_path, backup_dir / result_path.name)
print(f'v4r6 litmus files backed up to {backup_dir}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
`torch_dtype` is deprecated! Use `dtype` instead!
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).

Loading weights: 100%|██████████| 398/398 [00:34<00:00, 11.53it/s]

=== t0p0_s0 ===
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[1/12] read=True penalty=0.0 Why is the sky blue?
[2/12] read=False penalty=0.55 How do plants make their own food?
[3/12] read=False penalty=1.28 Why do we have day and night?
[4/12] read=True penalty=0.0 What makes ice melt?
[5/12] read=False penalty=0.52 How do magnets work?
[6/12] read=True penalty=0.0 Why do things fall to the ground?
[7/12] read=True penalty=0.0 Where does a puddle go when it dries up?
[8/12] read=False penalty=0.61 Why do we have seasons?
[9/12] read=True

## Stop after v4r6 development litmus

Copy both v4r6 litmus JSON files into local `eval/` for accuracy-v2 judging. Do not run `blind_v4r5`.

## Controlled v4r7 clean-union capacity run

v4r6 reached the strongest tuned accuracy-v2 result (10/12) but only 5/12 readability. This run keeps the clean benchmark-preserving data policy, expands to all 485 unique eligible records, and transfers v4r4's successful readability recipe: r32/a64, three epochs, and learning rate `2e-4`.

In [ ]:
%pip install -q --force-reinstall --no-cache-dir --no-deps pyarrow==18.1.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 MB 294.2 MB/s eta 0:00:00a 0:00:01


In [2]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')
repo_dir = Path('/content/SmallLearningModel_v4r7')
if not repo_dir.exists():
    subprocess.run([
        'git', 'clone', '--branch', 'main', '--single-branch',
        'https://github.com/Elitelord/SmallLearningModel.git', str(repo_dir),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo_dir), 'reset', '--hard', 'origin/main'], check=True)
os.chdir(repo_dir)
dataset_path = Path('data/v4/gold_v4_r7.jsonl')
stats_path = Path('data/v4/gold_v4_r7.stats.json')
if not dataset_path.exists() or not stats_path.exists():
    raise FileNotFoundError('Push the validated v4r7 dataset and stats before training')
stats = json.loads(stats_path.read_text(encoding='utf-8'))
dataset_bytes = dataset_path.read_bytes().replace(b'\r\n', b'\n').replace(b'\r', b'\n')
dataset_hash = hashlib.sha256(dataset_bytes).hexdigest()
if dataset_hash != stats.get('dataset_sha256'):
    raise ValueError(f'v4r7 dataset hash mismatch: {dataset_hash}')
try:
    import unsloth, trl, datasets, textstat
except ImportError:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'unsloth', 'trl', 'datasets', 'textstat',
    ], check=True)
except ValueError as error:
    if 'pyarrow' not in str(error).lower() and 'IpcReadOptions' not in str(error):
        raise
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
        '--no-cache-dir', '--no-deps', 'pyarrow==18.1.0',
    ], check=True)
    raise RuntimeError('PyArrow repaired. Restart the kernel, then rerun this cell.') from error
subprocess.run([
    sys.executable, '-m', 'data.audit_v4r3', str(dataset_path),
    '--accuracy-gate', 'clean-v2', '--forbid-targeted-v4r3',
], check=True)
adapter_dir = Path('train/adapters/v4r7')
if adapter_dir.exists():
    raise FileExistsError(f'{adapter_dir} already exists; refusing to overwrite it')
command = [
    sys.executable, '-m', 'train.qlora_train',
    '--data', str(dataset_path), '--adapter-name', 'v4r7',
    '--epochs', '3', '--lora-r', '32', '--lora-alpha', '64',
    '--batch-size', '8', '--grad-accum', '2', '--lr', '2e-4',
]
print('Starting v4r7 training.', flush=True)
process = subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, command)
backup_dir = Path('/content/drive/MyDrive/SmallLearningModel/v4r7')
backup_dir.mkdir(parents=True, exist_ok=True)
shutil.copytree(adapter_dir, backup_dir / 'adapter', dirs_exist_ok=True)
shutil.copy2(dataset_path, backup_dir / dataset_path.name)
shutil.copy2(stats_path, backup_dir / stats_path.name)
print(f'v4r7 adapter and dataset backed up to {backup_dir}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
Starting v4r7 training.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessi

In [3]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')
repo_dir = Path('/content/SmallLearningModel_v4r7')
if not repo_dir.exists():
    subprocess.run([
        'git', 'clone', '--branch', 'main', '--single-branch',
        'https://github.com/Elitelord/SmallLearningModel.git', str(repo_dir),
    ], check=True)
os.chdir(repo_dir)
backup_dir = Path('/content/drive/MyDrive/SmallLearningModel/v4r7')
adapter_dir = Path('train/adapters/v4r7')
if not adapter_dir.exists():
    shutil.copytree(backup_dir / 'adapter', adapter_dir, dirs_exist_ok=True)
output_path = Path('eval/v4r7_decode_calibration.json')
top_path = Path('eval/v4r7_decode_calibration.top.json')
for local_path in (output_path, top_path):
    drive_path = backup_dir / local_path.name
    if drive_path.exists() and not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_path, local_path)
if output_path.exists() and top_path.exists():
    print('v4r7 calibration files already exist; skipping duplicate generation.')
else:
    command = [
        sys.executable, '-u', '-m', 'eval.tuned_sweep',
        '--adapter', str(adapter_dir), '--eval-key', 'calibration_v4r5',
        '--temperatures', '0,0.3,0.7', '--seeds', '0,1,2',
        '--top-settings', '7', '--out', str(output_path),
    ]
    print('Starting v4r7 calibration.', flush=True)
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
for result_path in (output_path, top_path):
    shutil.copy2(result_path, backup_dir / result_path.name)
print(f'v4r7 calibration files backed up to {backup_dir}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Starting v4r7 calibration.
`torch_dtype` is deprecated! Use `dtype` instead!
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).

Loading weights: 100%|██████████| 398/398 [00:34<00:00, 11.41it/s]

=== t0p0_s0 ===
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[1/24] read=False penalty=0.39 What makes a rubber ball bounce?
[2/24] read=True penalty=0.0 How does soap lift dirt from our hands?
[3/24] read=False penalty=0.21 Why does metal feel colder than wood?
[4/24] read=True penalty=0.0 How does a thermos keep a drink warm?
[5/24] read=True penalty=0.0 How do bicycle brakes slow the wheels?
[6/24] read=False penalty=2.11 What makes static electricity build up?
[7/24] read=True penalty=0.0 How does glue hold 

## v4r7 calibration passed

Temperature `0` reached 17/24 readability. Temperatures `0.3` and `0.7` each averaged 15.33/24 across three seeds. The fixed development-litmus run below therefore uses temperature `0` and seed `0`. Keep `blind_v4r5` sealed.

In [4]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')
repo_dir = Path('/content/SmallLearningModel_v4r7')
os.chdir(repo_dir)
backup_dir = Path('/content/drive/MyDrive/SmallLearningModel/v4r7')
adapter_dir = Path('train/adapters/v4r7')
if not adapter_dir.exists():
    shutil.copytree(backup_dir / 'adapter', adapter_dir, dirs_exist_ok=True)
output_path = Path('eval/v4r7_decode_litmus.json')
top_path = Path('eval/v4r7_decode_litmus.top.json')
for local_path in (output_path, top_path):
    drive_path = backup_dir / local_path.name
    if drive_path.exists() and not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_path, local_path)
if output_path.exists() and top_path.exists():
    print('v4r7 litmus files already exist; refusing a duplicate run.')
else:
    command = [
        sys.executable, '-u', '-m', 'eval.tuned_sweep',
        '--adapter', str(adapter_dir),
        '--eval-key', 'eval_litmus', '--final-eval',
        '--temperatures', '0', '--seeds', '0',
        '--top-settings', '1', '--out', str(output_path),
    ]
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
for result_path in (output_path, top_path):
    shutil.copy2(result_path, backup_dir / result_path.name)
print(f'v4r7 litmus files backed up to {backup_dir}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
`torch_dtype` is deprecated! Use `dtype` instead!
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).

Loading weights: 100%|██████████| 398/398 [00:34<00:00, 11.50it/s]

=== t0p0_s0 ===
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[1/12] read=True penalty=0.0 Why is the sky blue?
[2/12] read=True penalty=0.0 How do plants make their own food?
[3/12] read=True penalty=0.0 Why do we have day and night?
[4/12] read=True penalty=0.0 What makes ice melt?
[5/12] read=True penalty=0.0 How do magnets work?
[6/12] read=True penalty=0.0 Why do things fall to the ground?
[7/12] read=True penalty=0.0 Where does a puddle go when it dries up?
[8/12] read=True penalty=0.0 Why do we have seasons?
[9/12] read=False penalt

## Stop after v4r7 development litmus

Copy both v4r7 litmus JSON files into local `eval/` for accuracy-v2 judging. Do not run `blind_v4r5`.

## Controlled v4r8 two-epoch midpoint

v4r7 reached 10/12 readability, 9/12 accuracy-v2, and 7/12 overall-v2. This controlled run changes only training duration from three epochs to two while keeping the same 485 clean records, r32/a64, learning rate `2e-4`, batching, and prompt.

In [5]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')
repo_dir = Path('/content/SmallLearningModel_v4r8')
if not repo_dir.exists():
    subprocess.run([
        'git', 'clone', '--branch', 'main', '--single-branch',
        'https://github.com/Elitelord/SmallLearningModel.git', str(repo_dir),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo_dir), 'reset', '--hard', 'origin/main'], check=True)
os.chdir(repo_dir)
dataset_path = Path('data/v4/gold_v4_r7.jsonl')
stats_path = Path('data/v4/gold_v4_r7.stats.json')
if not dataset_path.exists() or not stats_path.exists():
    raise FileNotFoundError('Push the validated v4r7 dataset and stats before training')
stats = json.loads(stats_path.read_text(encoding='utf-8'))
dataset_bytes = dataset_path.read_bytes().replace(b'\r\n', b'\n').replace(b'\r', b'\n')
dataset_hash = hashlib.sha256(dataset_bytes).hexdigest()
if dataset_hash != stats.get('dataset_sha256'):
    raise ValueError(f'v4r7 dataset hash mismatch: {dataset_hash}')
try:
    import unsloth, trl, datasets, textstat
except ImportError:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'unsloth', 'trl', 'datasets', 'textstat',
    ], check=True)
except ValueError as error:
    if 'pyarrow' not in str(error).lower() and 'IpcReadOptions' not in str(error):
        raise
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
        '--no-cache-dir', '--no-deps', 'pyarrow==18.1.0',
    ], check=True)
    raise RuntimeError('PyArrow repaired. Restart the kernel, then rerun this cell.') from error
subprocess.run([
    sys.executable, '-m', 'data.audit_v4r3', str(dataset_path),
    '--accuracy-gate', 'clean-v2', '--forbid-targeted-v4r3',
], check=True)
adapter_dir = Path('train/adapters/v4r8')
if adapter_dir.exists():
    raise FileExistsError(f'{adapter_dir} already exists; refusing to overwrite it')
command = [
    sys.executable, '-m', 'train.qlora_train',
    '--data', str(dataset_path), '--adapter-name', 'v4r8',
    '--epochs', '2', '--lora-r', '32', '--lora-alpha', '64',
    '--batch-size', '8', '--grad-accum', '2', '--lr', '2e-4',
]
print('Starting v4r8 training.', flush=True)
process = subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, command)
backup_dir = Path('/content/drive/MyDrive/SmallLearningModel/v4r8')
backup_dir.mkdir(parents=True, exist_ok=True)
shutil.copytree(adapter_dir, backup_dir / 'adapter', dirs_exist_ok=True)
shutil.copy2(dataset_path, backup_dir / dataset_path.name)
shutil.copy2(stats_path, backup_dir / stats_path.name)
print(f'v4r8 adapter and dataset backed up to {backup_dir}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Starting v4r8 training.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be diff

In [6]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')
repo_dir = Path('/content/SmallLearningModel_v4r8')
if not repo_dir.exists():
    subprocess.run([
        'git', 'clone', '--branch', 'main', '--single-branch',
        'https://github.com/Elitelord/SmallLearningModel.git', str(repo_dir),
    ], check=True)
os.chdir(repo_dir)
backup_dir = Path('/content/drive/MyDrive/SmallLearningModel/v4r8')
adapter_dir = Path('train/adapters/v4r8')
if not adapter_dir.exists():
    shutil.copytree(backup_dir / 'adapter', adapter_dir, dirs_exist_ok=True)
output_path = Path('eval/v4r8_decode_calibration.json')
top_path = Path('eval/v4r8_decode_calibration.top.json')
for local_path in (output_path, top_path):
    drive_path = backup_dir / local_path.name
    if drive_path.exists() and not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_path, local_path)
if output_path.exists() and top_path.exists():
    print('v4r8 calibration files already exist; skipping duplicate generation.')
else:
    command = [
        sys.executable, '-u', '-m', 'eval.tuned_sweep',
        '--adapter', str(adapter_dir), '--eval-key', 'calibration_v4r5',
        '--temperatures', '0,0.3,0.7', '--seeds', '0,1,2',
        '--top-settings', '7', '--out', str(output_path),
    ]
    print('Starting v4r8 calibration.', flush=True)
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
for result_path in (output_path, top_path):
    shutil.copy2(result_path, backup_dir / result_path.name)
print(f'v4r8 calibration files backed up to {backup_dir}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Starting v4r8 calibration.
`torch_dtype` is deprecated! Use `dtype` instead!
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).

Loading weights: 100%|██████████| 398/398 [00:34<00:00, 11.64it/s]

=== t0p0_s0 ===
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[1/24] read=False penalty=0.31 What makes a rubber ball bounce?
[2/24] read=True penalty=0.0 How does soap lift dirt from our hands?
[3/24] read=True penalty=0.0 Why does metal feel colder than wood?
[4/24] read=True penalty=0.0 How does a thermos keep a drink warm?
[5/24] read=True penalty=0.0 How do bicycle brakes slow the wheels?
[6/24] read=False penalty=2.09 What makes static electricity build up?
[7/24] read=False penalty=0.29 How does glue hold 

## v4r8 calibration passed

Temperature `0` reached 18/24 readability. Temperature `0.3` averaged 12.67/24 and `0.7` averaged 14.33/24 across three seeds. The fixed development-litmus run below therefore uses temperature `0` and seed `0`. Keep `blind_v4r5` sealed.

In [7]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')
repo_dir = Path('/content/SmallLearningModel_v4r8')
os.chdir(repo_dir)
backup_dir = Path('/content/drive/MyDrive/SmallLearningModel/v4r8')
adapter_dir = Path('train/adapters/v4r8')
if not adapter_dir.exists():
    shutil.copytree(backup_dir / 'adapter', adapter_dir, dirs_exist_ok=True)
output_path = Path('eval/v4r8_decode_litmus.json')
top_path = Path('eval/v4r8_decode_litmus.top.json')
for local_path in (output_path, top_path):
    drive_path = backup_dir / local_path.name
    if drive_path.exists() and not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_path, local_path)
if output_path.exists() and top_path.exists():
    print('v4r8 litmus files already exist; refusing a duplicate run.')
else:
    command = [
        sys.executable, '-u', '-m', 'eval.tuned_sweep',
        '--adapter', str(adapter_dir),
        '--eval-key', 'eval_litmus', '--final-eval',
        '--temperatures', '0', '--seeds', '0',
        '--top-settings', '1', '--out', str(output_path),
    ]
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
for result_path in (output_path, top_path):
    shutil.copy2(result_path, backup_dir / result_path.name)
print(f'v4r8 litmus files backed up to {backup_dir}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
`torch_dtype` is deprecated! Use `dtype` instead!
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).

Loading weights: 100%|██████████| 398/398 [00:33<00:00, 11.87it/s]

=== t0p0_s0 ===
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[1/12] read=True penalty=0.0 Why is the sky blue?
[2/12] read=False penalty=0.1 How do plants make their own food?
[3/12] read=True penalty=0.0 Why do we have day and night?
[4/12] read=True penalty=0.0 What makes ice melt?
[5/12] read=True penalty=0.0 How do magnets work?
[6/12] read=True penalty=0.0 Why do things fall to the ground?
[7/12] read=True penalty=0.0 Where does a puddle go when it dries up?
[8/12] read=True penalty=0.0 Why do we have seasons?
[9/12] read=False penal

## Stop after v4r8 development litmus

Copy both v4r8 litmus JSON files into local `eval/` for accuracy-v2 judging. Do not run `blind_v4r5`.

## Controlled v4r9 1.5-epoch ablation

v4r8 reached 9/12 readability, 9/12 accuracy-v2, and 8/12 overall-v2. This controlled run changes only training duration from two epochs to 1.5 while keeping the same 485 clean records, r32/a64, learning rate `2e-4`, batching, prompt, and evaluation policy.

In [1]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')
repo_dir = Path('/content/SmallLearningModel_v4r9')
if not repo_dir.exists():
    subprocess.run([
        'git', 'clone', '--branch', 'main', '--single-branch',
        'https://github.com/Elitelord/SmallLearningModel.git', str(repo_dir),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo_dir), 'reset', '--hard', 'origin/main'], check=True)
os.chdir(repo_dir)
dataset_path = Path('data/v4/gold_v4_r7.jsonl')
stats_path = Path('data/v4/gold_v4_r7.stats.json')
if not dataset_path.exists() or not stats_path.exists():
    raise FileNotFoundError('Push the validated v4r7 dataset and stats before training')
stats = json.loads(stats_path.read_text(encoding='utf-8'))
dataset_bytes = dataset_path.read_bytes().replace(b'\r\n', b'\n').replace(b'\r', b'\n')
dataset_hash = hashlib.sha256(dataset_bytes).hexdigest()
if dataset_hash != stats.get('dataset_sha256'):
    raise ValueError(f'v4r7 dataset hash mismatch: {dataset_hash}')
try:
    import unsloth, trl, datasets, textstat
except ImportError:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'unsloth', 'trl', 'datasets', 'textstat',
    ], check=True)
except ValueError as error:
    if 'pyarrow' not in str(error).lower() and 'IpcReadOptions' not in str(error):
        raise
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
        '--no-cache-dir', '--no-deps', 'pyarrow==18.1.0',
    ], check=True)
    raise RuntimeError('PyArrow repaired. Restart the kernel, then rerun this cell.') from error
subprocess.run([
    sys.executable, '-m', 'data.audit_v4r3', str(dataset_path),
    '--accuracy-gate', 'clean-v2', '--forbid-targeted-v4r3',
], check=True)
adapter_dir = Path('train/adapters/v4r9')
if adapter_dir.exists():
    raise FileExistsError(f'{adapter_dir} already exists; refusing to overwrite it')
command = [
    sys.executable, '-m', 'train.qlora_train',
    '--data', str(dataset_path), '--adapter-name', 'v4r9',
    '--epochs', '1.5', '--lora-r', '32', '--lora-alpha', '64',
    '--batch-size', '8', '--grad-accum', '2', '--lr', '2e-4',
]
print('Starting v4r9 training.', flush=True)
process = subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, command)
backup_dir = Path('/content/drive/MyDrive/SmallLearningModel/v4r9')
backup_dir.mkdir(parents=True, exist_ok=True)
shutil.copytree(adapter_dir, backup_dir / 'adapter', dirs_exist_ok=True)
shutil.copy2(dataset_path, backup_dir / dataset_path.name)
shutil.copy2(stats_path, backup_dir / stats_path.name)
print(f'v4r9 adapter and dataset backed up to {backup_dir}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
Starting v4r9 training.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessi

In [2]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')
repo_dir = Path('/content/SmallLearningModel_v4r9')
if not repo_dir.exists():
    subprocess.run([
        'git', 'clone', '--branch', 'main', '--single-branch',
        'https://github.com/Elitelord/SmallLearningModel.git', str(repo_dir),
    ], check=True)
os.chdir(repo_dir)
backup_dir = Path('/content/drive/MyDrive/SmallLearningModel/v4r9')
adapter_dir = Path('train/adapters/v4r9')
if not adapter_dir.exists():
    shutil.copytree(backup_dir / 'adapter', adapter_dir, dirs_exist_ok=True)
output_path = Path('eval/v4r9_decode_calibration.json')
top_path = Path('eval/v4r9_decode_calibration.top.json')
for local_path in (output_path, top_path):
    drive_path = backup_dir / local_path.name
    if drive_path.exists() and not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_path, local_path)
if output_path.exists() and top_path.exists():
    print('v4r9 calibration files already exist; skipping duplicate generation.')
else:
    command = [
        sys.executable, '-u', '-m', 'eval.tuned_sweep',
        '--adapter', str(adapter_dir), '--eval-key', 'calibration_v4r5',
        '--temperatures', '0,0.3,0.7', '--seeds', '0,1,2',
        '--top-settings', '7', '--out', str(output_path),
    ]
    print('Starting v4r9 calibration.', flush=True)
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
for result_path in (output_path, top_path):
    shutil.copy2(result_path, backup_dir / result_path.name)
print(f'v4r9 calibration files backed up to {backup_dir}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Starting v4r9 calibration.
`torch_dtype` is deprecated! Use `dtype` instead!
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).

Loading weights: 100%|██████████| 398/398 [00:33<00:00, 11.97it/s]

=== t0p0_s0 ===
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[1/24] read=False penalty=0.28 What makes a rubber ball bounce?
[2/24] read=True penalty=0.0 How does soap lift dirt from our hands?
[3/24] read=True penalty=0.0 Why does metal feel colder than wood?
[4/24] read=False penalty=0.69 How does a thermos keep a drink warm?
[5/24] read=False penalty=0.25 How do bicycle brakes slow the wheels?
[6/24] read=False penalty=0.0 What makes static electricity build up?
[7/24] read=True penalty=0.0 How does glue hold

## v4r9 fixed calibration policy

The next cell chooses temperature using only `calibration_v4r5`: highest mean readability passes across seeds, then lowest mean penalty, then lowest temperature. It fixes seed 0 for the one development-litmus run and saves the choice before inference.

In [3]:
from collections import defaultdict
from google.colab import drive
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')
repo_dir = Path('/content/SmallLearningModel_v4r9')
os.chdir(repo_dir)
backup_dir = Path('/content/drive/MyDrive/SmallLearningModel/v4r9')
adapter_dir = Path('train/adapters/v4r9')
if not adapter_dir.exists():
    shutil.copytree(backup_dir / 'adapter', adapter_dir, dirs_exist_ok=True)
calibration_path = Path('eval/v4r9_decode_calibration.json')
if not calibration_path.exists():
    drive_calibration = backup_dir / calibration_path.name
    if not drive_calibration.exists():
        raise FileNotFoundError('Run the v4r9 calibration cell first')
    calibration_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_calibration, calibration_path)
calibration = json.loads(calibration_path.read_text(encoding='utf-8'))
by_temperature = defaultdict(list)
for setting in calibration['settings']:
    by_temperature[float(setting['temperature'])].append(setting)
ranked_temperatures = []
for temperature, settings in by_temperature.items():
    ranked_temperatures.append({
        'temperature': temperature,
        'seed': 0,
        'mean_readability_passes': sum(item['readability_passes'] for item in settings) / len(settings),
        'mean_penalty': sum(item['mean_penalty'] for item in settings) / len(settings),
        'calibration_settings': len(settings),
    })
ranked_temperatures.sort(key=lambda item: (
    -item['mean_readability_passes'], item['mean_penalty'], item['temperature'],
))
selection = ranked_temperatures[0]
selection_path = Path('eval/v4r9_decode_selection.json')
selection_path.write_text(json.dumps({
    'policy': 'max mean readability, min mean penalty, min temperature; fixed seed 0',
    'selected': selection,
    'ranked_temperatures': ranked_temperatures,
}, indent=2) + '\n', encoding='utf-8')
shutil.copy2(selection_path, backup_dir / selection_path.name)
temperature = f"{selection['temperature']:g}"
print(f'Selected temperature={temperature}, seed=0 from calibration only.', flush=True)
output_path = Path('eval/v4r9_decode_litmus.json')
top_path = Path('eval/v4r9_decode_litmus.top.json')
for local_path in (output_path, top_path):
    drive_path = backup_dir / local_path.name
    if drive_path.exists() and not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_path, local_path)
if output_path.exists() and top_path.exists():
    print('v4r9 litmus files already exist; refusing a duplicate run.')
else:
    command = [
        sys.executable, '-u', '-m', 'eval.tuned_sweep',
        '--adapter', str(adapter_dir),
        '--eval-key', 'eval_litmus', '--final-eval',
        '--temperatures', temperature, '--seeds', '0',
        '--top-settings', '1', '--out', str(output_path),
    ]
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
for result_path in (output_path, top_path):
    shutil.copy2(result_path, backup_dir / result_path.name)
print(f'v4r9 litmus files backed up to {backup_dir}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Selected temperature=0.7, seed=0 from calibration only.
`torch_dtype` is deprecated! Use `dtype` instead!
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).

Loading weights: 100%|██████████| 398/398 [00:33<00:00, 11.75it/s]

=== t0p7_s0 ===
[1/12] read=False penalty=0.18 Why is the sky blue?
[2/12] read=False penalty=0.21 How do plants make their own food?
[3/12] read=True penalty=0.0 Why do we have day and night?
[4/12] read=True penalty=0.0 What makes ice melt?
[5/12] read=False penalty=0.41 How do magnets work?
[6/12] read=True penalty=0.0 Why do things fall to the ground?
[7/12] read=False penalty=0.35 Where does a puddle go when it dries up?
[8/12] read=False penalty=0.39 Why do we have seasons?
[9/12] read=False penalty=0.73 How do our lungs help us breathe?
[10/12] read=False pe

## Stop after v4r9 development litmus

Copy the v4r9 calibration, selection, and litmus JSON files into local `eval/` for accuracy-v2 judging. Do not run `blind_v4r5`.

In [ ]:
import sys, torch
print(sys.executable)
print("GPU:", torch.cuda.is_available())

In [2]:
from pathlib import Path
import os, subprocess, sys

repo = Path("/content/SmallLearningModel_demo")

if not repo.exists():
    subprocess.run([
        "git", "clone",
        "https://github.com/Elitelord/SmallLearningModel.git",
        str(repo),
    ], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)

os.chdir(repo)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", "demo/requirements.txt", "bitsandbytes",
], check=True)

env = os.environ.copy()
env["MODEL_ID"] = "SAgarwal34/qwen3-4b-grade3-science-v4r8"
env["GRADIO_SHARE"] = "1"
env["PYTHONUNBUFFERED"] = "1"

subprocess.run(
    [sys.executable, "-u", "demo/app.py"],
    env=env,
    check=True,
)

KeyboardInterrupt: 

In [3]:
import os, subprocess, sys

os.chdir("/content/SmallLearningModel_demo")
env = os.environ.copy()
env["MODEL_ID"] = "SAgarwal34/qwen3-4b-grade3-science-v4r8"
env["GRADIO_SHARE"] = "1"

subprocess.run([sys.executable, "-u", "demo/app.py"], env=env, check=True)

KeyboardInterrupt: 